In [0]:
# Databricks notebook source
# GOLD LAYER: Circuit Performance Analysis
# Purpose: Team/driver strengths and weaknesses by circuit type

from pyspark.sql.functions import (
    col, sum as spark_sum, avg, count, when, lit,
    round as spark_round, max as spark_max, min as spark_min,
    dense_rank, current_timestamp, countDistinct
)
from pyspark.sql.window import Window

# ============ 1. READ FROM SILVER ============
fact = spark.read.table("f1_data.silver.fact_results")

race_results = fact.filter(col("session_type").isin("Race", "Sprint"))

# ============ 2. DRIVER PERFORMANCE BY CIRCUIT ============
driver_circuit = race_results.groupBy(
    "driver_number",
    "driver_name",
    "team_name",
    "team_colour",
    "year",
    "meeting_key",
    "meeting_name",
    "circuit_short_name",
    "circuit_type",
    "race_country",
    "race_location"
).agg(
    spark_sum("points").alias("total_points"),
    avg("position").alias("avg_finish_position_raw"),
    spark_min("position").alias("best_finish"),
    count("session_key").alias("times_raced_here"),
    spark_sum(when(col("position") == 1, 1).otherwise(0)).alias("wins_here"),
    spark_sum(when(col("position") <= 3, 1).otherwise(0)).alias("podiums_here"),
    spark_sum(when(col("dnf") == True, 1).otherwise(0)).alias("dnfs_here")
) \
.withColumn("avg_finish_position",
    spark_round(col("avg_finish_position_raw"), 2)
) \
.drop("avg_finish_position_raw")

# ============ 3. TEAM PERFORMANCE BY CIRCUIT TYPE ============
# Street, Permanent, Hybrid circuits perform differently for each team
team_circuit_type = race_results.groupBy(
    "team_name",
    "circuit_type",
    "year"
).agg(
    spark_sum("points").alias("total_points"),
    avg("position").alias("avg_position_raw"),
    count("session_key").alias("races_on_circuit_type"),
    spark_sum(when(col("position") == 1, 1).otherwise(0)).alias("wins"),
    spark_sum(when(col("position") <= 3, 1).otherwise(0)).alias("podiums")
) \
.withColumn("avg_position", spark_round(col("avg_position_raw"), 2)) \
.drop("avg_position_raw") \
.withColumn("circuit_type_strength",
    # Which circuit type does this team perform best on?
    when(col("avg_position") <= 3, lit("DOMINANT"))
    .when(col("avg_position") <= 6, lit("STRONG"))
    .when(col("avg_position") <= 10, lit("COMPETITIVE"))
    .otherwise(lit("WEAK"))
) \
.withColumn("gold_processed_at", current_timestamp()) \
.withColumn("gold_source", lit("f1_data.silver.fact_results"))

# ============ 4. WRITE TO GOLD ============
driver_circuit \
    .withColumn("gold_processed_at", current_timestamp()) \
    .withColumn("gold_source", lit("f1_data.silver.fact_results")) \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year") \
    .saveAsTable("f1_data.gold.circuit_performance")

team_circuit_type.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year") \
    .saveAsTable("f1_data.gold.team_circuit_strengths")

print(f"✓ gold.circuit_performance written: {driver_circuit.count()} records")
print(f"✓ gold.team_circuit_strengths written: {team_circuit_type.count()} records")

✓ gold.circuit_performance written: 359 records
✓ gold.team_circuit_strengths written: 30 records
